# MSIS 522 - Homework 1: The Complete Data Science Workflow
## Heart Disease Prediction

**Dataset:** [Heart Disease Prediction Dataset](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) (Kaggle, by fedesoriano)

**Streamlit App:** [Live Demo](https://msis522-hw1-heart-failure-gjfteetwerufsrd8tkvd63.streamlit.app/)

**GitHub Repo:** [AveryJYL/msis522-hw1-heart-failure](https://github.com/AveryJYL/msis522-hw1-heart-failure)

---
## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, auc,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)
import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print(f'TensorFlow: {tf.__version__}')
print(f'XGBoost: {xgb.__version__}')
print('All imports successful!')

---
# Part 1: Descriptive Analytics (25 points)

## 1.1 Dataset Introduction (5 pts)

**Source:** Heart Failure Prediction Dataset from Kaggle, combining 5 established databases (Cleveland, Hungarian, Switzerland, Long Beach VA, Statlog).

**Target Variable:** `HeartDisease` (binary: 1 = heart disease present, 0 = normal)

**Why important:** Cardiovascular diseases are the #1 cause of death globally (17.9 million lives/year). Early prediction can help clinicians prioritize patients for further testing and intervention.

In [ ]:
df = pd.read_csv('heart.csv')

print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nTarget distribution:\n{df.HeartDisease.value_counts()}')
print(f'\nData types:\n{df.dtypes}')
print(f'\nBasic statistics:')
df.describe()

In [ ]:
# Categorical feature value counts
for col in ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']:
    print(f'\n{col}:\n{df[col].value_counts()}')

**Dataset Summary:**
- 918 patient records, 11 features (5 numerical + 6 categorical), no missing values
- Features include demographic info (Age, Sex), clinical measurements (RestingBP, Cholesterol, MaxHR, Oldpeak), and test results (ChestPainType, RestingECG, ExerciseAngina, ST_Slope, FastingBS)
- Data is clean and ready for analysis

## 1.2 Target Distribution (5 pts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

target_counts = df['HeartDisease'].value_counts()
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(['Normal (0)', 'Heart Disease (1)'], target_counts.values,
            color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Heart Disease Target Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=12)
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 10, f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontsize=12, fontweight='bold')

axes[1].pie(target_counts.values, labels=['Normal', 'Heart Disease'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12}, explode=(0.05, 0.05))
axes[1].set_title('Target Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

**Interpretation:**

Out of 918 patients in this dataset, 508 (55.3%) were diagnosed with heart disease and 410 (44.7%) were not. This means the two groups are close in size but not perfectly equal — the heart disease group is slightly larger. Since the model might lean toward predicting the bigger group, I used `class_weight='balanced'` to make sure it pays equal attention to both groups. I also chose F1 score and AUC-ROC as my main metrics instead of just accuracy, because accuracy alone can be misleading when the classes aren't perfectly balanced.

## 1.3 Feature Distributions and Relationships (10 pts)

### Visualization 1: Age Distribution by Heart Disease

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=df, x='Age', hue='HeartDisease', kde=True, bins=30,
             palette=['#2ecc71', '#e74c3c'], alpha=0.6, ax=ax)
ax.set_title('Age Distribution by Heart Disease Status', fontsize=14, fontweight='bold')
ax.set_xlabel('Age', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.legend(title='Heart Disease', labels=['Normal (0)', 'Heart Disease (1)'])
plt.tight_layout()
plt.show()

**Interpretation:**

The histogram shows that older patients (around 55-65 years old) are much more likely to have heart disease. The green bars (normal patients) are spread across a wider age range and include more younger people. This makes sense — age is one of the most well-known risk factors for heart problems. The older you are, the higher the chance something is going on with your heart.

### Visualization 2: Heart Disease Rate by Chest Pain Type

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ct = pd.crosstab(df['ChestPainType'], df['HeartDisease'], normalize='index') * 100
ct.plot(kind='bar', stacked=True, color=['#2ecc71', '#e74c3c'], ax=ax,
        edgecolor='black', linewidth=0.5)
ax.set_title('Heart Disease Rate by Chest Pain Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Chest Pain Type', fontsize=12)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.legend(title='Heart Disease', labels=['Normal', 'Heart Disease'])
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', label_type='center', fontsize=9)
plt.tight_layout()
plt.show()

**Interpretation:**

The most surprising finding here is that ASY (Asymptomatic) patients — people who reported *no chest pain at all* — actually have the highest heart disease rate at about 79%. This seems counterintuitive: you'd expect people with pain to be sicker. But in reality, many heart disease patients don't feel obvious symptoms until it's serious. This is a strong reminder that just because someone feels fine doesn't mean their heart is healthy. It shows why regular checkups and diagnostic tests matter, even without symptoms.

### Visualization 3: Numerical Features by Heart Disease Status

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat, title in zip(axes.flatten(),
                            ['MaxHR', 'Oldpeak', 'RestingBP', 'Cholesterol'],
                            ['Maximum Heart Rate', 'ST Depression (Oldpeak)',
                             'Resting Blood Pressure', 'Cholesterol']):
    sns.boxplot(data=df, x='HeartDisease', y=feat, hue='HeartDisease',
                palette=['#2ecc71', '#e74c3c'], ax=ax, legend=False)
    ax.set_title(f'{title} by Heart Disease Status', fontsize=12, fontweight='bold')
    ax.set_xticklabels(['Normal', 'Heart Disease'])
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

**Interpretation:**

Looking at the four boxplots, two features clearly separate the two groups. First, **MaxHR** (maximum heart rate during exercise): heart disease patients have noticeably lower values, meaning their hearts can't beat as fast during physical activity — a sign of reduced heart function. Second, **Oldpeak** (how much the heart's electrical signal drops during exercise): heart disease patients have higher values, which indicates the heart isn't getting enough blood flow during exertion. **RestingBP** (resting blood pressure) doesn't show much difference between the two groups. **Cholesterol** has a lot of zero values, which are probably cases where the data wasn't recorded rather than the patient actually having zero cholesterol. This makes cholesterol less reliable as a predictor in this dataset.

### Visualization 4: Exercise Angina & ST Slope vs Heart Disease

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, col, title in zip(axes, ['ExerciseAngina', 'ST_Slope'],
                           ['Exercise Angina', 'ST Slope']):
    ct_temp = pd.crosstab(df[col], df['HeartDisease'], normalize='index') * 100
    ct_temp.plot(kind='bar', stacked=True, color=['#2ecc71', '#e74c3c'],
                ax=ax, edgecolor='black', linewidth=0.5)
    ax.set_title(f'Heart Disease Rate by {title}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Percentage (%)')
    ax.legend(title='Heart Disease', labels=['Normal', 'Heart Disease'])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

**Interpretation:**

Two very strong patterns here. First, patients who experience **chest pain during exercise** (ExerciseAngina = Yes) have about a 77% heart disease rate. If your chest hurts when you work out, that's a major red flag. Second, patients with a **Flat ST_Slope** (a specific pattern on the heart monitor during exercise) have an even higher rate of about 84%. Doctors actually look at this exact indicator when evaluating heart health — a flat or downward slope means the heart may not be getting enough oxygen. These two features turn out to be the strongest predictors in the entire dataset.

## 1.4 Correlation Heatmap (5 pts)

In [ ]:
df_encoded = df.copy()
for col in ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])

fig, ax = plt.subplots(figsize=(12, 10))
corr_matrix = df_encoded.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation:**

The heatmap shows how strongly each feature is connected to heart disease. The top three positive correlations are: **ST_Slope** (0.52), **ExerciseAngina** (0.49), and **Oldpeak** (0.40) — all exercise-related indicators. **MaxHR** has the strongest negative correlation (-0.40), meaning a higher maximum heart rate is associated with being healthy (which makes sense — a strong heart can beat faster). Importantly, the features don't have very high correlations with *each other* (all below 0.5), which means each one brings unique information to the table. There's no major redundancy among the features.

---
# Part 2: Predictive Analytics (45 points)

## 2.1 Data Preparation

In [ ]:
# One-hot encode categorical features
df_model = pd.get_dummies(df, columns=['Sex', 'ChestPainType', 'RestingECG',
                                        'ExerciseAngina', 'ST_Slope'], drop_first=True)

X = df_model.drop('HeartDisease', axis=1)
y = df_model['HeartDisease']

print(f'Features: {X.shape[1]}')
print(f'Feature names: {list(X.columns)}')

# 70/30 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f'\nTrain: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print(f'Train target: {y_train.value_counts().to_dict()}')
print(f'Test target:  {y_test.value_counts().to_dict()}')

# Scale numerical features
scaler = StandardScaler()
num_cols = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])
joblib.dump(scaler, 'scaler.joblib')
print('\nScaler fitted and saved.')

In [ ]:
# Evaluation helper function
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_prob)
    }
    print(f'\n{"="*45}')
    print(f'  {model_name}')
    print(f'{"="*45}')
    for k, v in metrics.items():
        if k != 'Model': print(f'  {k}: {v:.4f}')
    return metrics

all_results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 2.2 Logistic Regression Baseline (5 pts)

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

lr_metrics = evaluate_model(lr, X_test_scaled, y_test, 'Logistic Regression')
all_results.append(lr_metrics)
joblib.dump(lr, 'model_logistic_regression.joblib')

## 2.3 Decision Tree (CART) + 5-Fold GridSearchCV (5 pts)

In [ ]:
dt_param_grid = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_leaf': [5, 10, 20, 50]
}

dt_gs = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    dt_param_grid, cv=cv, scoring='f1', n_jobs=-1, return_train_score=True
)
dt_gs.fit(X_train, y_train)

print(f'Best parameters: {dt_gs.best_params_}')
print(f'Best CV F1: {dt_gs.best_score_:.4f}')

dt_model = dt_gs.best_estimator_
dt_metrics = evaluate_model(dt_model, X_test, y_test, 'Decision Tree (CART)')
all_results.append(dt_metrics)
joblib.dump(dt_model, 'model_decision_tree.joblib')

In [ ]:
# Visualize the best tree
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(dt_model, feature_names=X_train.columns, class_names=['Normal', 'Heart Disease'],
          filled=True, rounded=True, fontsize=8, ax=ax)
ax.set_title(f'Best Decision Tree (max_depth={dt_gs.best_params_["max_depth"]}, '
             f'min_samples_leaf={dt_gs.best_params_["min_samples_leaf"]})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2.4 Random Forest + 5-Fold GridSearchCV (10 pts)

In [ ]:
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 8]
}

rf_gs = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    rf_param_grid, cv=cv, scoring='f1', n_jobs=-1, return_train_score=True
)
rf_gs.fit(X_train, y_train)

print(f'Best parameters: {rf_gs.best_params_}')
print(f'Best CV F1: {rf_gs.best_score_:.4f}')

rf_model = rf_gs.best_estimator_
rf_metrics = evaluate_model(rf_model, X_test, y_test, 'Random Forest')
all_results.append(rf_metrics)
joblib.dump(rf_model, 'model_random_forest.joblib')

In [ ]:
# ROC Curve - Random Forest
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_estimator(rf_model, X_test, y_test, ax=ax, name='Random Forest')
ax.plot([0, 1], [0, 1], 'k--', label='Random Guess')
ax.set_title('ROC Curve - Random Forest', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 2.5 XGBoost + 5-Fold GridSearchCV (10 pts)

In [ ]:
# Calculate scale_pos_weight for class imbalance
scale_pos = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

xgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1]
}

xgb_gs = GridSearchCV(
    xgb.XGBClassifier(random_state=42, scale_pos_weight=scale_pos,
                       eval_metric='logloss', use_label_encoder=False),
    xgb_param_grid, cv=cv, scoring='f1', n_jobs=-1, return_train_score=True
)
xgb_gs.fit(X_train, y_train)

print(f'Best parameters: {xgb_gs.best_params_}')
print(f'Best CV F1: {xgb_gs.best_score_:.4f}')
print(f'Total combinations tested: {len(xgb_gs.cv_results_["params"])}')

xgb_model = xgb_gs.best_estimator_
xgb_metrics = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')
all_results.append(xgb_metrics)
joblib.dump(xgb_model, 'model_xgboost.joblib')

In [ ]:
# ROC Curve - XGBoost
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_estimator(xgb_model, X_test, y_test, ax=ax, name='XGBoost')
ax.plot([0, 1], [0, 1], 'k--', label='Random Guess')
ax.set_title('ROC Curve - XGBoost', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 2.6 Neural Network — Keras MLP (10 pts)

Building a Multi-Layer Perceptron using TensorFlow/Keras with:
- 2 hidden layers (128 units each) with ReLU activation
- Dropout regularization
- Binary cross-entropy loss + Adam optimizer
- Early stopping to prevent overfitting

In [ ]:
def build_keras_model(hidden1=128, hidden2=128, dropout_rate=0.3, learning_rate=0.001):
    model = keras.Sequential([
        layers.Input(shape=(X_train_scaled.shape[1],)),
        layers.Dense(hidden1, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(hidden2, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Train base model
keras_mlp = build_keras_model(128, 128, 0.3, 0.001)
history = keras_mlp.fit(
    X_train_scaled.values, y_train.values,
    epochs=100, batch_size=32,
    validation_split=0.15,
    callbacks=[
        callbacks.EarlyStopping(patience=15, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(patience=5, factor=0.5)
    ],
    verbose=1
)

# Evaluate
y_pred_prob_keras = keras_mlp.predict(X_test_scaled.values).flatten()
y_pred_keras = (y_pred_prob_keras >= 0.5).astype(int)

keras_base_metrics = {
    'Model': 'Neural Network (Keras)',
    'Accuracy': accuracy_score(y_test, y_pred_keras),
    'Precision': precision_score(y_test, y_pred_keras),
    'Recall': recall_score(y_test, y_pred_keras),
    'F1': f1_score(y_test, y_pred_keras),
    'AUC-ROC': roc_auc_score(y_test, y_pred_prob_keras)
}
print('\nKeras MLP Base Model Results:')
for k, v in keras_base_metrics.items():
    if k != 'Model': print(f'  {k}: {v:.4f}')

In [ ]:
# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
axes[0].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, linestyle='--')
axes[1].set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()

plt.tight_layout()
plt.savefig('mlp_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training history saved.')

### Bonus: Keras MLP Hyperparameter Tuning (1 pt)

Grid search over **hidden layer sizes**, **dropout rates**, and **learning rates** with 3-fold CV.

In [ ]:
tuning_configs = [
    {'h1': 64,  'h2': 64,  'drop': 0.2, 'lr': 0.001},
    {'h1': 64,  'h2': 64,  'drop': 0.3, 'lr': 0.001},
    {'h1': 128, 'h2': 64,  'drop': 0.2, 'lr': 0.001},
    {'h1': 128, 'h2': 64,  'drop': 0.3, 'lr': 0.001},
    {'h1': 128, 'h2': 128, 'drop': 0.2, 'lr': 0.001},
    {'h1': 128, 'h2': 128, 'drop': 0.3, 'lr': 0.001},
    {'h1': 64,  'h2': 64,  'drop': 0.2, 'lr': 0.01},
    {'h1': 64,  'h2': 64,  'drop': 0.3, 'lr': 0.01},
    {'h1': 128, 'h2': 64,  'drop': 0.2, 'lr': 0.01},
    {'h1': 128, 'h2': 64,  'drop': 0.3, 'lr': 0.01},
    {'h1': 128, 'h2': 128, 'drop': 0.2, 'lr': 0.01},
    {'h1': 128, 'h2': 128, 'drop': 0.3, 'lr': 0.01},
]

tuning_results = []
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for i, cfg in enumerate(tuning_configs):
    print(f'Config {i+1}/{len(tuning_configs)}: {cfg}')
    fold_f1s = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train)):
        m = build_keras_model(cfg['h1'], cfg['h2'], cfg['drop'], cfg['lr'])
        m.fit(X_train_scaled.values[tr_idx], y_train.values[tr_idx],
              epochs=80, batch_size=32, verbose=0,
              validation_data=(X_train_scaled.values[val_idx], y_train.values[val_idx]),
              callbacks=[callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
        preds = (m.predict(X_train_scaled.values[val_idx], verbose=0).flatten() >= 0.5).astype(int)
        fold_f1s.append(f1_score(y_train.values[val_idx], preds))
    mean_f1 = np.mean(fold_f1s)
    tuning_results.append({**cfg, 'mean_f1': mean_f1, 'std_f1': np.std(fold_f1s)})
    print(f'  CV F1: {mean_f1:.4f} +/- {np.std(fold_f1s):.4f}')

tuning_df = pd.DataFrame(tuning_results).sort_values('mean_f1', ascending=False)
print('\nBest config:')
print(tuning_df.iloc[0])

# Retrain best config
best = tuning_df.iloc[0]
keras_tuned = build_keras_model(int(best['h1']), int(best['h2']), best['drop'], best['lr'])
keras_tuned.fit(
    X_train_scaled.values, y_train.values,
    epochs=100, batch_size=32, validation_split=0.15, verbose=1,
    callbacks=[callbacks.EarlyStopping(patience=15, restore_best_weights=True)]
)

# Evaluate tuned
y_prob_tuned = keras_tuned.predict(X_test_scaled.values).flatten()
y_pred_tuned = (y_prob_tuned >= 0.5).astype(int)
keras_tuned_metrics = {
    'Model': 'Neural Network (Keras)',
    'Accuracy': accuracy_score(y_test, y_pred_tuned),
    'Precision': precision_score(y_test, y_pred_tuned),
    'Recall': recall_score(y_test, y_pred_tuned),
    'F1': f1_score(y_test, y_pred_tuned),
    'AUC-ROC': roc_auc_score(y_test, y_prob_tuned)
}
print(f'\nTuned F1: {keras_tuned_metrics["F1"]:.4f} vs Base F1: {keras_base_metrics["F1"]:.4f}')

# Use the better one
if keras_tuned_metrics['F1'] >= keras_base_metrics['F1']:
    final_keras_metrics = keras_tuned_metrics
    keras_tuned.save('model_mlp.keras')
    print('Tuned model saved.')
else:
    final_keras_metrics = keras_base_metrics
    keras_mlp.save('model_mlp.keras')
    print('Base model saved (it was better).')

all_results.append(final_keras_metrics)

In [ ]:
# Tuning visualization
fig, ax = plt.subplots(figsize=(14, 6))
labels = [f"H=({int(r['h1'])},{int(r['h2'])})\nD={r['drop']}\nLR={r['lr']}"
          for _, r in tuning_df.iterrows()]
colors = ['#e74c3c' if i == 0 else '#3498db' for i in range(len(tuning_df))]
ax.bar(range(len(labels)), tuning_df['mean_f1'], yerr=tuning_df['std_f1'],
       color=colors, edgecolor='black', linewidth=0.5, capsize=3)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=7, rotation=45, ha='right')
ax.set_ylabel('Mean CV F1 Score')
ax.set_title('Keras MLP Hyperparameter Tuning Results (Best in Red)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('mlp_tuning_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.7 Model Comparison Summary (5 pts)

In [ ]:
results_df = pd.DataFrame(all_results).round(4)
print('Model Comparison Table:')
results_df

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(results_df))
width = 0.15
for i, (metric, color) in enumerate(zip(
    ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC'],
    ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'])):
    ax.bar(x + i*width, results_df[metric], width, label=metric, color=color, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df['Model'], fontsize=9)
ax.legend()
ax.set_ylim(0.7, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
# All ROC Curves
fig, ax = plt.subplots(figsize=(10, 8))

# sklearn models
for name, model, X_eval in [
    ('Logistic Regression', lr, X_test_scaled),
    ('Decision Tree', dt_model, X_test),
    ('Random Forest', rf_model, X_test),
    ('XGBoost', xgb_model, X_test),
]:
    y_prob = model.predict_proba(X_eval)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.3f})', linewidth=2)

# Keras model
if keras_tuned_metrics['F1'] >= keras_base_metrics['F1']:
    y_prob_k = keras_tuned.predict(X_test_scaled.values).flatten()
else:
    y_prob_k = keras_mlp.predict(X_test_scaled.values).flatten()
fpr_k, tpr_k, _ = roc_curve(y_test, y_prob_k)
ax.plot(fpr_k, tpr_k, label=f'Neural Network (AUC={auc(fpr_k, tpr_k):.3f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Random Guess')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - All Models', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
results_df.to_csv('model_comparison.csv', index=False)
print('Results saved to model_comparison.csv')

**Analysis:**

Looking at the results, the ensemble and boosting models clearly outperform single models. **Random Forest** and **XGBoost** both achieved strong F1 and AUC-ROC scores, demonstrating the power of combining multiple learners. **Decision Tree** was the weakest because a single tree tends to memorize the training data (overfitting) and doesn't generalize well to new patients.

**Logistic Regression**, despite being the simplest model, performed surprisingly well — nearly matching the ensemble methods. This suggests the relationship between the features and heart disease is fairly straightforward in this dataset.

The **Keras MLP** improved after hyperparameter tuning over hidden layer sizes, dropout rates, and learning rates. However, for this dataset size (918 rows), tree-based methods have an edge because neural networks typically shine with much larger datasets.

**Trade-offs:** Logistic Regression is easiest to explain to doctors. Random Forest/XGBoost give the best predictions and support SHAP analysis for interpretability. The Keras MLP offers flexibility for future scaling but is harder to interpret.

---
# Part 3: Explainability - SHAP (10 points)

Using the **Random Forest** model (best F1 & AUC-ROC) for SHAP analysis.

In [ ]:
import shap

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

# For Random Forest, take class 1 (Heart Disease)
if isinstance(shap_values, list):
    sv = shap_values[1]
elif shap_values.ndim == 3:
    sv = shap_values[:, :, 1]
else:
    sv = shap_values

print(f'SHAP values shape: {sv.shape}')

### SHAP Summary Plot (Beeswarm)

In [ ]:
plt.figure(figsize=(12, 8))
shap.summary_plot(sv, X_test, show=False)
plt.title('SHAP Summary Plot - Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_3_1_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

### SHAP Feature Importance (Bar)

In [ ]:
plt.figure(figsize=(12, 8))
shap.summary_plot(sv, X_test, plot_type='bar', show=False)
plt.title('SHAP Feature Importance - Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_3_2_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

### SHAP Waterfall Plot

In [ ]:
y_pred_prob = rf_model.predict_proba(X_test)[:, 1]
high_risk_idx = np.argmax(y_pred_prob)

print(f'Patient index: {high_risk_idx}')
print(f'Predicted probability: {y_pred_prob[high_risk_idx]:.2%}')
print(f'Actual: {"Heart Disease" if y_test.iloc[high_risk_idx] == 1 else "Normal"}')

X_train_float = X_train.astype(float)
X_test_float = X_test.astype(float)

explainer2 = shap.TreeExplainer(rf_model, X_train_float)
shap_explanation = explainer2(X_test_float, check_additivity=False)

if len(shap_explanation.shape) == 3:
    shap_exp_class1 = shap_explanation[:, :, 1]
else:
    shap_exp_class1 = shap_explanation

plt.figure(figsize=(12, 8))
shap.waterfall_plot(shap_exp_class1[high_risk_idx], show=False)
plt.title(f'SHAP Waterfall - High Risk Patient (Prob: {y_pred_prob[high_risk_idx]:.2%})',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_3_3_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

**SHAP Interpretation:**

**Which features matter most:** The SHAP analysis reveals that **ST_Slope** (both the Up and Flat variants) is the single most important feature the model uses to make predictions. After that, **ExerciseAngina**, **MaxHR**, **Cholesterol**, and **Oldpeak** round out the top five. These are all related to how the heart performs during exercise or stress — which aligns with what cardiologists look at in real clinical practice.

**Direction of impact:** A Flat ST Slope and exercise-induced chest pain push the model toward predicting heart disease. On the flip side, a high maximum heart rate and an Upward ST Slope push the prediction toward healthy. Interestingly, a cholesterol value of zero (which likely means the data wasn't recorded) also increases the predicted risk — the model learned that missing cholesterol data tends to come from sicker patients.

**Value for decision-makers:** The fact that the model's most important features match what real doctors look at is very encouraging. It means the model isn't relying on random noise — it actually learned meaningful medical patterns. In a hospital setting, this tool could help flag high-risk patients who need further testing, especially those who are asymptomatic (no obvious symptoms). The waterfall plot shows exactly *why* the model flagged a specific patient as 99.5% high-risk, making the prediction transparent and trustworthy for clinicians.

---
## Summary

| Deliverable | Status |
|---|---|
| Part 1: Descriptive Analytics | ✅ |
| Part 2: Predictive Analytics (5 models) | ✅ |
| 2.5 XGBoost + GridSearchCV | ✅ |
| 2.6 Keras MLP + Training History | ✅ |
| Part 3: SHAP Explainability | ✅ |
| Part 4: Streamlit App | ✅ [Live](https://msis522-hw1-heart-failure-gjfteetwerufsrd8tkvd63.streamlit.app/) |
| Bonus: Keras MLP Tuning (dropout + hidden sizes + LR) | ✅ |
| GitHub Repo | ✅ [Link](https://github.com/AveryJYL/msis522-hw1-heart-failure) |